In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

from itertools import islice

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext

from google.genai import types
from typing import Optional,Dict,Any

from neo4j_for_adk import tool_success, tool_error, graphdb

from helper import make_agent_caller

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported
Libraries imported


In [2]:
load_dotenv()

MODEL_NAME = os.getenv("DEEPSEEK_MODEL")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(
    model=llm.model,
    messages=[{"role": "user", "content": "你准备好了吗"}],
    tools=[]))

print("\nDeepseek已经准备好了")

ModelResponse(id='883dbfe3-0945-4231-91ee-3449f21785ce', created=1788411300, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='准备好了！😊 我随时可以为你提供帮助，无论是回答问题、创作内容、查找信息，还是陪你聊天解闷，都没问题～  \n有什么想聊的或需要处理的，尽管告诉我吧！✨', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=44, prompt_tokens=7, total_tokens=51, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=7))

Deepseek已经准备好了


In [3]:
file_suggestion_agent_instruction = """
你是一个负责审查文件列表的建设性审查 AI。你的目标是推荐用于构建知识图谱的相关文件。

**任务：**
审查文件列表，评估它们与“已批准的用户目标”中所指定的图谱类型和描述是否相关。

对于任何你不确定的文件，请使用 'sample_file' 工具，以便更好地了解该文件的内容。

仅考虑结构化数据文件，如 CSV 或 JSON。

准备任务：
- 使用 'get_approved_user_goal' 工具获取已批准的用户目标

仔细思考，重复以下步骤直到完成：
1. 使用 'list_available_files' 工具列出可用文件
2. 评估每个文件的相关性，然后使用 'set_suggested_files' 工具记录推荐文件列表
3. 使用 'get_suggested_files' 工具获取推荐文件列表
4. 请求用户批准这组推荐文件
5. 如果用户有反馈意见，请牢记该反馈并返回步骤 1
6. 如果获得批准，使用 'approve_suggested_files' 工具记录该批准操作

"""

In [4]:
from helper import get_neo4j_import_dir
from tools import get_approved_user_goal

In [5]:
# 工具：列出导入文件 (List Import Files)

ALL_AVAILABLE_FILES = "all_available_files"

def list_available_files(tool_context:ToolContext) -> dict:
    f"""列出可用于构建知识图谱的文件。
    所有文件路径均相对于导入目录 (import directory)。

    返回:
        dict: 包含内容元数据的字典。
        包含一个 'status' 键（'success' 或 'error'）。
        如果状态为 'success'，则包含一个 '{ALL_AVAILABLE_FILES}' 键，其值为文件列表。
        如果状态为 'error'，则包含一个 'error_message' 键。
        'error_message' 可能包含关于如何处理错误的指示。
    """

    # 使用辅助函数获取导入目录
    import_dir = Path(get_neo4j_import_dir())

    # 获取相对文件名的列表，因此文件路径必须以导入目录为根节点
    file_names = [str(x.relative_to(import_dir))
                  for x in import_dir.rglob("*")
                  if x.is_file()]

    # 将该列表保存到状态 (state) 中，以便我们稍后检查它
    tool_context.state[ALL_AVAILABLE_FILES] = file_names

    return tool_success(ALL_AVAILABLE_FILES, file_names)

In [6]:
# 工具：对文件进行采样 (Sample File)
def sample_file(file_path: str, tool_context: ToolContext) -> dict:
    """通过将文件内容作为文本读取来进行采样。

    将任何文件都视为纯文本处理，并且最多只读取前 100 行。

    参数:
        file_path: 要采样的文件，路径必须是相对于导入目录的相对路径

    返回:
        dict: 一个包含内容元数据以及文件采样内容的字典。
              包含一个 'status' 键（值为 'success' 或 'error'）。
              如果状态为 'success'，则包含一个 'content' 键，值为文本格式的文件内容。
              如果状态为 'error'，则包含一个 'error_message' 键。
    """

    # 1. 绝对路径拦截检查
    if Path(file_path).is_absolute():
        return tool_error("文件路径必须是相对于导入目录的相对路径。请确保该文件来自于可用文件列表。")

    import_dir = Path(get_neo4j_import_dir())

    # 2. 拼接完整路径（仅对相对路径有效）
    full_path_to_file = import_dir / file_path

    # 3. 检查文件是否存在
    if not full_path_to_file.exists():
        return tool_error(f"导入目录中不存在该文件: {file_path}")

    try:
        # 4. 将所有文件视为文本读取
        with open(full_path_to_file, 'r', encoding='utf-8') as file:
            # 使用 islice 最多读取 100 行
            lines = list(islice(file, 100))
            content = ''.join(lines)
            return tool_success("content", content)

    except Exception as e:
        # 5. 错误捕获
        return tool_error(f"读取或处理文件 {file_path} 时出错: {e}")

In [7]:
from typing import List

SUGGESTED_FILES = "suggested_files"

def set_suggested_files(suggest_files: List[str], tool_context: ToolContext) -> dict:
    """设置用于数据导入的文件。
    """

    # 💡 增加防呆检查：检查列表是否为空
    # 如果 suggest_files 是 []，not suggest_files 就会变成 True
    if not suggest_files:
        # 给大模型返回明确的错误提示，指导它下一步该怎么做
        return tool_error("传入的文件列表为空！请先调用 list_available_files 工具查找系统中的有效文件，然后再执行本操作。")
    # 将传入的文件列表保存到 Agent 的状态 (state) 中
    tool_context.state[SUGGESTED_FILES] = suggest_files
    return tool_success(SUGGESTED_FILES, suggest_files)

# 有助于鼓励大语言模型 (LLM) 首先设置建议的文件
def get_suggested_files(tool_context: ToolContext) -> Dict[str, Any]:
    """获取用于数据导入的文件。
    """
    # 从 Agent 的状态 (state) 中读取并返回之前保存的文件列表
    return tool_success(SUGGESTED_FILES, tool_context.state[SUGGESTED_FILES])

In [8]:
APPROVED_FILES = "approved_files"

def approve_suggested_files(tool_context: ToolContext) -> Dict[str, Any]:
    """批准状态 (state) 中的 {SUGGESTED_FILES} (建议文件)，
    以便将其作为 {APPROVED_FILES} (已批准文件) 进行后续处理。

    如果 {SUGGESTED_FILES} 不在状态 (state) 中，则返回一个错误。
    """

    # 防呆设计：检查大模型是否跳过了“建议”步骤直接来批准
    if SUGGESTED_FILES not in tool_context.state:
        # 截图结尾处的 "Take no action o..." 意为 "不要采取任何操作"
        return tool_error("当前文件尚未设置。请勿执行操作...")

    # 将“建议文件”的内容复制给“已批准文件”，完成状态的转正升级
    tool_context.state[APPROVED_FILES] = tool_context.state[SUGGESTED_FILES]

    return tool_success(APPROVED_FILES, tool_context.state[APPROVED_FILES])

In [9]:
# 用于“文件推荐智能体 (file suggestion agent)”的工具列表
file_suggestion_agent_tools = [get_approved_user_goal, list_available_files, sample_file, set_suggested_files, get_suggested_files, approve_suggested_files]

In [ ]:
# 智能体构建

In [11]:
file_suggestion_agent = Agent(
    name="file_suggestion_agent_v1",
    model=llm,
    description="帮助用户选择要导入的文件",
    instruction=file_suggestion_agent_instruction,
    tools=file_suggestion_agent_tools,
)

print(f"智能体'{file_suggestion_agent.name}'已创建")

智能体'file_suggestion_agent_v1'已创建


In [20]:
from helper import make_agent_caller

file_suggestion_caller = await make_agent_caller(agent=file_suggestion_agent, initial_state={
     "approved_user_goal": {
        "kind_of_graph": "供应链分析", # 翻译：图谱类型：
        "description": "用于制造产品的多级物料清单" # 翻译：描述：用于制造产品的多级物料清单...
    }
})

async def run_conversation():
   await file_suggestion_caller.chat("我们可以使用哪些文件进行导入？")

await run_conversation()

# await file_suggestion_caller.chat("我们可以使用哪些文件进行导入？")

# 获取对话结束后的会话状态（记忆）
session_end = await file_suggestion_caller.get_session()

print("\n---\n")

# 使用 .get() 方法，如果找不到键，就返回后面的默认提示语，防止程序崩溃
print("可用文件: ", session_end.state.get(ALL_AVAILABLE_FILES, "⚠️ Agent 并没有将可用文件存入 State，请检查它是否调用了工具！"))
print("建议文件: ", session_end.state.get(SUGGESTED_FILES, "⚠️ Agent 并没有推荐文件！"))




>>>👤 用户: 我们可以使用哪些文件进行导入？
<<< Agent Response: 智能体没有生成最终响应


---

可用文件:  []
建议文件:  ⚠️ Agent 并没有推荐文件！
